# Video Generation

 An imange is a 2-D tensor. A video is a 3-D one. The theory is the same; The compute is 10-100x harder. 

 ## Problem definition

 A 10-second 1080p video at 24fps is 240 frames of 1920x1080x3 pixels. That's ~1.5GB of raw data per clip. Pixel-space diffusion is infeasible. You need:
 1. Spatiotemporal compression. A VAE that encodes videos, not frames, into a sequence of spatial-temporal patches.
 2. Temporal coherence.  Frames need to share content, lighting, and object indentity over seconds. The net has to model motion.
 3. Compute budget. Video trainig is 10-100x more expensive than image for same model size.
 4. Conditioning. Text, image(first-frame), audio, or another video. Most production models accept all four.

 The architecture that solved this is the Diffusion Transformer applied to spatiotemporal patches, trained on huge (prompt, caption, video) datasets, Same diffusion loss.

 ## Basic Concept

 Video diffusion: patchify, DiT, decode

 ### Patchify

 Encode the video with a 3D VAE (learned spatiotemporal compression). That latent is shape `(T_latent, H_latent, M_latent, C_latent)`. Split into patches of size `[t_p, h_p, w_p]`. 

 ### Spatiotemporal DiT

 A transfomer processes the flat sequence of patches. Each patch has a 3D positional embedding `(time + y + x)`. Attention is usually factorized:
 * Spatial attention.  within each frame's patches.
 * Temporal attention. across frames at the same spatial location.
 * Full 3D attention is 16-100x more expensive. Used ony at low resolution or in research.

 ### Text conditioning

 Cross-attention with a large text encoder.

 Long prompts matters -- Sora's training set had GPT-generated dense re-captions averaging 200 tokens per clip.

 ### Training

 Standard diffusion loss `(epsilon or v prediction)` over spatiotemporal latents.

# Build your Own

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(21)

T_FRAMES, POS_DIM = 6, 4
T, T_DIM, HIDDEN = 100, 8, 48
STEPS, BATCH, LR = 3000, 64, 1e-2


class VideoDenoiser(nn.Module):
    """Predict noise for all frames jointly (sees full clip + frame position)."""

    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x, t_emb):
        return self.net(torch.cat([x, t_emb], dim=-1))


def sin_embed(t, dim):
    """Sinusoidal embedding. t: scalar / (B,) / (B, L)."""
    if not torch.is_tensor(t):
        t = torch.tensor(t, dtype=torch.float32)
    t = t.float()
    half = dim // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, device=t.device).float() / max(half - 1, 1)))
    # broadcast: (..., 1) * (half,) -> (..., half)
    angles = t.unsqueeze(-1) * freqs
    return torch.cat([angles.sin(), angles.cos()], dim=-1)[..., :dim]


def make_schedule(steps):
    betas = torch.linspace(1e-4, 0.08, steps)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars


def make_video(n):
    """Batch of 1-D 'videos': smooth trajectories of length T_FRAMES."""
    base = torch.randn(n, 1)
    slope = 0.3 * torch.randn(n, 1)
    t = torch.arange(T_FRAMES).float().view(1, -1)
    return base + slope * t + 0.05 * torch.randn(n, T_FRAMES)


def patchify_with_pos(video):
    """Each frame value + its time position embedding → flat vector per clip."""
    # video: (B, T)
    b = video.shape[0]
    pe = sin_embed(torch.arange(T_FRAMES), POS_DIM)  # (T, POS_DIM)
    pe = pe.unsqueeze(0).expand(b, -1, -1)           # (B, T, POS_DIM)
    patches = torch.cat([video.unsqueeze(-1), pe], dim=-1)  # (B, T, 1+POS)
    return patches.reshape(b, -1)


betas, alphas, alpha_bars = make_schedule(T)
in_dim = T_FRAMES * (1 + POS_DIM)
net = VideoDenoiser(in_dim + T_DIM, HIDDEN, T_FRAMES)
opt = torch.optim.Adam(net.parameters(), lr=LR)
loss_fn = nn.MSELoss()

print(f"=== training joint video DDPM: {T_FRAMES} frames per clip ===")
print(f"schedule alpha_bar[-1]={alpha_bars[-1].item():.4f}")
net.train()
for step in range(1, STEPS + 1):
    video = make_video(BATCH)
    t = torch.randint(0, T, (BATCH,))
    eps = torch.randn_like(video)
    abar = alpha_bars[t].unsqueeze(-1)
    noisy = abar.sqrt() * video + (1.0 - abar).sqrt() * eps
    x = patchify_with_pos(noisy)
    loss = loss_fn(net(x, sin_embed(t, T_DIM)), eps)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 1000 == 0:
        print(f"  step {step}: loss {loss.item():.4f}")


@torch.no_grad()
def sample_joint(n=1):
    x = torch.randn(n, T_FRAMES)
    for t in range(T - 1, -1, -1):
        t_batch = torch.full((n,), t, dtype=torch.long)
        eps_hat = net(patchify_with_pos(x), sin_embed(t_batch, T_DIM))
        mean = (x - betas[t] / (1.0 - alpha_bars[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        x = mean if t == 0 else mean + betas[t].sqrt() * torch.randn_like(x)
    return x


def independent_per_frame(n=1):
    """Baseline: each frame independent (flickers)."""
    return torch.randn(n, T_FRAMES) + 0.3 * torch.arange(T_FRAMES).float()


def frame_deltas(video):
    return (video[:, 1:] - video[:, :-1]).abs()


print()
print("=== 5 clips, joint sampling (coherent) ===")
net.eval()
joint_clips = sample_joint(5)
joint_deltas = frame_deltas(joint_clips)
for i, clip in enumerate(joint_clips):
    print("  clip {}: ".format(i) + " ".join(f"{v:+.2f}" for v in clip.tolist()))

print()
print("=== 5 clips, independent per-frame (flicker baseline) ===")
indep_clips = independent_per_frame(5)
indep_deltas = frame_deltas(indep_clips)
for i, clip in enumerate(indep_clips):
    print("  clip {}: ".format(i) + " ".join(f"{v:+.2f}" for v in clip.tolist()))

avg_joint = joint_deltas.mean().item()
avg_indep = indep_deltas.mean().item()
print()
print(f"avg frame-to-frame delta: joint={avg_joint:.2f}  independent={avg_indep:.2f}")
print("joint sampling produces smoother motion (smaller deltas).")
